In [0]:
%sql
--Definir catalogo
use catalog curso_eng_dados;
use schema modulo_um;

In [0]:
import requests
import json
from pyspark.sql.functions import (current_timestamp, lit)
import pandas as pd



# Igestão de dados via API
url = "https://randomuser.me/api/?results=100"

response = requests.get(url)

dados = response.json()

print(f"Quantidade de registros: {len(dados)}")
#print(dados)

json_strings = [json.dumps(x) for x in dados["results"]]

bronze_df = spark.createDataFrame(
    [(x,) for x in json_strings],
    ["payload"]
)

total_linhas = bronze_df.count()

bronze_df = bronze_df \
    .withColumn("dt_ingestao", current_timestamp()) \
    .withColumn("source", lit("RandomUser")) \
    .withColumn("pipeline", lit("api_random"))

#1a execução
#bronze_df.write.mode("overwrite").saveAsTable("curso_eng_dados.modulo_um.raw_randomuser_data")

#Demais cargas
bronze_df.write.mode("append").saveAsTable("curso_eng_dados.modulo_um.raw_randomuser_data")